# Topic 9: Hash Maps — O(1) Everything

## 9.1 How Hash Maps Work Internally

### 📊 [VISUAL] Hash Map Internal Mechanism

```
When you do:  d['name'] = 'Alice'

  1. Python computes:  hash('name') -> 7234891234  (integer)
  2. Maps to bucket:   7234891234 % table_size -> bucket 3
  3. Stores:           bucket[3] -> [('name', 'Alice')]

When you do:  d['name']  (lookup)
  1. hash('name') -> same number -> same bucket
  2. Find ('name','Alice') in bucket 3
  Total: O(1) average

COLLISION: two keys hash to same bucket
  Resolution: Chaining (linked list in bucket) or Open Addressing

LOAD FACTOR = n_elements / n_buckets
  Python resizes dict when load factor > 2/3 -> O(1) amortised

KEY RULE: only hashable (immutable) types can be dict keys
  int, float, str, tuple  -> hashable    (can be keys)
  list, dict, set         -> NOT hashable (cannot be keys)
```

---

## 9.2 Essential Hash Map Patterns

In [ ]:
from collections import Counter, defaultdict

# ── PATTERN 1: Frequency Count — O(n) ────────────────────────────────────────
def word_frequency(text):
    words = text.lower().split()
    return Counter(words)

freq = word_frequency('the cat sat on the mat the cat')
print("Top 2:", freq.most_common(2))   # [('the',3), ('cat',2)]


# ── PATTERN 2: Two Sum — O(n) using hash map ─────────────────────────────────
def two_sum(nums, target):
    seen = {}                          # value -> index
    for i, num in enumerate(nums):
        complement = target - num
        if complement in seen:         # O(1) lookup
            return [seen[complement], i]
        seen[num] = i
    return []

print("Two sum:", two_sum([2, 7, 11, 15], 9))   # [0, 1]


# ── PATTERN 3: Group Anagrams — O(n * k log k) ───────────────────────────────
def group_anagrams(words):
    groups = defaultdict(list)
    for word in words:
        key = ''.join(sorted(word))    # sorted chars as canonical key
        groups[key].append(word)
    return list(groups.values())

print("Anagrams:", group_anagrams(['eat','tea','tan','ate','nat','bat']))


# ── PATTERN 4: Sliding Window + Hash Map — longest unique substring ───────────
def longest_unique_substring(s):
    char_index = {}                    # char -> last seen index
    max_len = start = 0
    for i, ch in enumerate(s):
        if ch in char_index and char_index[ch] >= start:
            start = char_index[ch] + 1  # move window start
        char_index[ch] = i
        max_len = max(max_len, i - start + 1)
    return max_len

print("Longest unique in 'abcabcbb':", longest_unique_substring('abcabcbb'))  # 3
print("Longest unique in 'pwwkew':  ", longest_unique_substring('pwwkew'))    # 3


# ── PATTERN 5: Memoisation cache ─────────────────────────────────────────────
def memoize(func):
    cache = {}
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
    return wrapper

@memoize
def expensive_compute(n):
    return sum(i**2 for i in range(n))

print("Cached compute(100):", expensive_compute(100))

## 9.3 Advanced Python Hash Map Tools

In [ ]:
from collections import Counter, defaultdict, OrderedDict

# ── Counter: frequency counting powerhouse ───────────────────────────────────
c = Counter('abracadabra')
print(c)                      # Counter({'a':5,'b':2,'r':2,'c':1,'d':1})
print(c.most_common(3))       # [('a',5),('b',2),('r',2)]

# Arithmetic on Counters (useful for comparing distributions)
c1 = Counter(a=3, b=2)
c2 = Counter(a=1, b=4)
print("c1 - c2:", c1 - c2)   # Counter({'a':2}) — only positives kept


# ── defaultdict: never KeyError ──────────────────────────────────────────────
d_list = defaultdict(list)
d_list['group1'].append('Alice')    # no KeyError — default is []
d_list['group1'].append('Bob')

d_int = defaultdict(int)
for ch in 'hello':
    d_int[ch] += 1              # default is 0
print("Char freq:", dict(d_int))


# ── LRU Cache — OrderedDict ───────────────────────────────────────────────────
class LRUCache:
    """Least Recently Used cache — O(1) get and put."""

    def __init__(self, capacity):
        self.cache    = OrderedDict()
        self.capacity = capacity

    def get(self, key):
        if key not in self.cache: return -1
        self.cache.move_to_end(key)      # mark as recently used
        return self.cache[key]

    def put(self, key, value):
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.capacity:
            self.cache.popitem(last=False)  # evict LRU (oldest = front)

    def __repr__(self): return f'LRU({dict(self.cache)})'


lru = LRUCache(3)
lru.put(1, 'A'); lru.put(2, 'B'); lru.put(3, 'C')
print("get(1):", lru.get(1))    # 'A'  — moves 1 to most recent
lru.put(4, 'D')                 # evicts key 2 (least recently used)
print("get(2):", lru.get(2))    # -1  (evicted)
print(lru)

> 🔵 **[AI/ML]** Hash maps are used in almost every part of AI/ML infrastructure:
> - **NLP Vocabulary:** `word → integer index` mapping — `tokenizer.vocab` in HuggingFace Transformers IS a hash map.
> - **Feature Hashing (the hashing trick):** map high-cardinality features to fixed-size arrays using `hash(feature) % n_buckets` — used in sklearn's `HashingVectorizer`.
> - **Caching inference results:** LRU cache for expensive model predictions — if input seen before, return cached output O(1) vs O(model_forward_pass).
> - **Gradient accumulation:** `{param_name: accumulated_gradient}` dict in custom trainers.

---

## ✏️ Exercises — Hash Maps

**[EXERCISE 9.1 — Easy]** Write `find_duplicates(arr)` that returns a list of all elements that appear more than once. Use a `Counter`. O(n). Example: `[1,2,3,2,4,3,5]` → `[2,3]`

**[EXERCISE 9.2 — Medium]** Write `is_isomorphic(s, t)` that returns `True` if string `s` can be mapped to string `t` character-by-character (bijective mapping). Example: `'egg','add'` → True; `'foo','bar'` → False; `'paper','title'` → True


------